# Umbral de confianza, recall por imagen, falsas alarmas y mAP común (iteración 1)

Calcula las métricas que Ultralytics no reporta y que `criterios_exito_provisional.md` (apartado 1.1) pone como principales:

- **Umbral de operación**: el más alto que todavía conserva un recall de cajas de 0.90 en validación.
- **Recall por imagen**: de las fotos con oso, en cuántas el modelo pone al menos una detección por encima del umbral. Meta: 0.95 o más.
- **Falsas alarmas**: de las fotos sin oso, cuántas reciben alguna detección por encima del umbral. Meta: menos del 10 %, por fuente.
- Lo mismo separando día y noche en ENA24.

Usa solo el conjunto de validación, salvo que se active `EVALUAR_PRUEBA` en la celda 0.

**Versión 2 (2 de octubre de 2026).** Los modelos a evaluar se eligen por nombre en la celda 0. Reconoce los de YOLO26 (`e<N>[b]_<tamaño>_deg<D>_s<S>`), los de la ablación por fuente (`a<N>[b]_<tamaño>_<fuente>_s<S>`), sus `last.pt` si se subieron como `<corrida>_last.pt`, y el modelo COCO sin entrenar (`coco_n_sinentrenar`), del que solo se usa la clase `bear` con el argumento `classes` de Ultralytics. Si un modelo no alcanza el recall de cajas de 0.90, su umbral queda vacío y se informa como inalcanzable, en lugar de mostrar ceros.

**Versión 3 (6 de octubre de 2026).**
- Evalúa también modelos que no son de Ultralytics a partir de sus detecciones exportadas (tipo `externo`). Por ahora, RT-DETRv3-R18 (`e<N>[b]_rtdetrv3r18_s<S>`), con los archivos `<RUN_ID>_detecciones_val.csv.gz` que deja `rtdetrv3_entrenar.ipynb`. Sus detecciones se emparejan y se miden igual que las de YOLO26.
- Calcula mAP50 y mAP50-95 con el mismo código para todos los modelos: el evaluador oficial de COCO (`pycocotools` 2.0.11, la versión que usó PaddleDetection, con sus parámetros estándar). Es la cifra que se compara entre arquitecturas (pregunta 32). Como control cruzado, calcula el mismo mAP con el emparejamiento de este cuaderno y `compute_ap` de Ultralytics, y lo compara con lo que reportó cada librería.
- Comprueba que E1 y E1b repitan las cifras de septiembre y que pycocotools repita, sobre las detecciones exportadas de RT-DETRv3, el mAP50-95 que dio PaddleDetection.
- Aplica la regla de D25 con los parámetros fijados en la celda 0 antes de correrlo.
- El resto no cambia. El emparejamiento ahora deja de recorrer las detecciones de una foto cuando ya no le quedan cajas libres, lo que no cambia ningún resultado y lo hace más rápido.

Configuración del cuaderno: GPU T4, **Internet activado** (para instalar Ultralytics y pycocotools), y los datasets `deteccionoso` (imágenes), `yolodeteccionoso` (pesos de YOLO26, en su versión más reciente) y el de las detecciones de RT-DETRv3 adjuntos.

## 0. Parámetros

In [1]:
UMBRAL_SOBRE    = "ena24"   # partición de validación donde se exige el recall de 0.90: "ena24" o "global"
RECALL_OBJETIVO = 0.90      # criterios de éxito, apartado 1.1
IOU_MIN         = 0.5       # una detección acierta si su caja se solapa al menos 0.5 (IoU) con una caja real
CONF_MIN        = 0.001     # se guardan todas las detecciones desde esta confianza, igual que la validación de Ultralytics
EVALUAR_PRUEBA  = False     # dejar en False: el conjunto de prueba se usa una sola vez, con los modelos finales

# Modelos a evaluar, por nombre de corrida (None = todos los que se encuentren, más el modelo COCO).
# E1 y E1b van como control: deben repetir las cifras de septiembre. E4 y E4b (YOLO26m) y E6 y E6b
# (RT-DETRv3-R18, desde sus detecciones exportadas) son la comparación emparejada en parámetros.
INCLUIR = ["e1_n_deg0_s0", "e1b_n_deg0_s1", "e4_m_deg0_s0", "e4b_m_deg0_s1",
           "e6_rtdetrv3r18_s0", "e6b_rtdetrv3r18_s1", "e7_m_deg0_s0"]
COCO_PESOS = "yolo26n.pt"   # pesos COCO originales; Ultralytics los descarga si no están (hace falta Internet)
COCO_CLASE = 21             # 'bear' en COCO; la celda 4 comprueba el nombre antes de usarla

# Regla de D25, fijada antes de correr este cuaderno (condición 2 precisada el 6 de octubre de 2026):
# se entrena YOLO26m con 200 épocas si RT-DETRv3 tiene su mejor época en la EPOCA_MAX_D25 o antes con sus dos
# semillas, y si la media de su mAP50-95 global de validación (pycocotools) supera a la media de YOLO26m
# por más de MARGEN_D25. Si no, se comparan a 100 épocas y se declara.
GRUPO_RTDETR  = ["e6_rtdetrv3r18_s0", "e6b_rtdetrv3r18_s1"]
GRUPO_YOLO26M = ["e4_m_deg0_s0", "e4b_m_deg0_s1"]
EPOCA_MAX_D25 = 90
MARGEN_D25    = 0.015

assert UMBRAL_SOBRE in ("ena24", "global")
print(f"Umbral elegido sobre validación '{UMBRAL_SOBRE}' con recall de cajas >= {RECALL_OBJETIVO}, IoU >= {IOU_MIN}")
print("Conjuntos a evaluar:", ["val", "test"] if EVALUAR_PRUEBA else ["val"])
print("Modelos pedidos:", "todos" if INCLUIR is None else ", ".join(INCLUIR))
print(f"D25: mejor época <= {EPOCA_MAX_D25} y media de {GRUPO_RTDETR} por encima de la de {GRUPO_YOLO26M} "
      f"por más de {MARGEN_D25}")

Umbral elegido sobre validación 'ena24' con recall de cajas >= 0.9, IoU >= 0.5
Conjuntos a evaluar: ['val']
Modelos pedidos: e1_n_deg0_s0, e1b_n_deg0_s1, e4_m_deg0_s0, e4b_m_deg0_s1, e6_rtdetrv3r18_s0, e6b_rtdetrv3r18_s1, e7_m_deg0_s0
D25: mejor época <= 90 y media de ['e6_rtdetrv3r18_s0', 'e6b_rtdetrv3r18_s1'] por encima de la de ['e4_m_deg0_s0', 'e4b_m_deg0_s1'] por más de 0.015


## 1. Entorno

In [2]:
%pip install -q ultralytics==8.4.138 pycocotools==2.0.11

import sys, torch, ultralytics, numpy, PIL
from importlib.metadata import version
print("Python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("Ultralytics :", ultralytics.__version__)
print("numpy       :", numpy.__version__, "| Pillow:", PIL.__version__, "| pandas:", version("pandas"))
print("pycocotools :", version("pycocotools"))

Note: you may need to restart the kernel to use updated packages.
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Python      : 3.13.16
torch       : 2.11.0+cu130 | CUDA: True
GPU         : Tesla T4
Ultralytics : 8.4.138
numpy       : 2.1.3 | Pillow: 13.0.0 | pandas: 2.3.3
pycocotools : 2.0.11


## 2. Rutas y modelos

Busca los archivos `*_best.pt` y `*_last.pt` dentro de los datasets adjuntos y deduce de cada nombre el tipo de modelo, el tamaño, con qué datos se entrenó y la semilla. Busca también las detecciones exportadas de RT-DETRv3 (`<RUN_ID>_detecciones_val.csv.gz`) y su `<RUN_ID>_best.json`, de donde salen la mejor época y el mAP50-95 que dio PaddleDetection. Añade el modelo COCO sin entrenar. Los nombres pedidos en `INCLUIR` que no aparecen se avisan y se omiten. Los archivos con nombres que no siguen ninguna convención se listan aparte y no se usan.

In [3]:
import glob, json, re
from pathlib import Path

DATASET = Path("/kaggle/input/datasets/josemartinvallejos/deteccionoso")
assert DATASET.is_dir(), f"No existe {DATASET}. Revisar que el dataset de imágenes esté adjunto."

TAMANO = {"n": "nano", "s": "small", "m": "medium"}
ENTRENADO_CON = {None: "ENA24 + iNaturalist", "ena24": "solo ENA24", "inat": "solo iNaturalist"}
PATRONES = [
    re.compile(r"^(?P<corrida>e\d+b?)_(?P<tam>[nsm])_deg(?P<deg>\d+)_s(?P<semilla>\d+)$"),
    re.compile(r"^(?P<corrida>a\d+b?)_(?P<tam>[nsm])_(?P<fuente>ena24|inat)_s(?P<semilla>\d+)$"),
]
PATRON_EXTERNO = re.compile(r"^(?P<corrida>e\d+b?)_rtdetrv3r18_s(?P<semilla>\d+)$")   # RT-DETRv3-R18 (PaddleDetection)
PATRON_ARCHIVO_DET = re.compile(r"^(?P<base>.+)_detecciones_(?P<conj>val|test)\.csv(\.gz)?$")

def describir_pesos(ruta):
    """Datos del modelo según el nombre del archivo, o None si no sigue ninguna convención."""
    for sufijo in ("_best", "_last"):
        if ruta.stem.endswith(sufijo):
            base = ruta.stem[: -len(sufijo)]
            break
    else:
        return None
    for patron in PATRONES:
        m = patron.match(base)
        if m:
            g = m.groupdict()
            return {"run_id": base if sufijo == "_best" else base + "_last",
                    "tipo": "yolo26" if g["corrida"].startswith("e") else "ablacion",
                    "tamano": TAMANO[g["tam"]], "entrenado_con": ENTRENADO_CON[g.get("fuente")],
                    "degrees": int(g.get("deg") or 0), "semilla": int(g["semilla"]),
                    "pesos": sufijo[1:], "ruta": ruta, "clases": None}
    return None

encontrados, ignorados = {}, []
for p in sorted(Path(x) for x in glob.glob("/kaggle/input/**/*.pt", recursive=True)):
    d = describir_pesos(p)
    if d is None:
        ignorados.append(p)
    elif d["run_id"] in encontrados:
        print(f"AVISO: {d['run_id']} aparece dos veces; se usa {encontrados[d['run_id']]['ruta']} y se ignora {p}")
    else:
        encontrados[d["run_id"]] = d

# Detecciones exportadas por otra librería (tipo "externo"): <RUN_ID>_detecciones_<val|test>.csv(.gz)
for p in sorted(Path(x) for x in glob.glob("/kaggle/input/**/*_detecciones_*.csv*", recursive=True)):
    m = PATRON_ARCHIVO_DET.match(p.name)
    g = PATRON_EXTERNO.match(m.group("base")) if m else None
    if g is None:
        ignorados.append(p)
        continue
    rid, conj = m.group("base"), m.group("conj")
    if rid in encontrados and encontrados[rid]["tipo"] != "externo":
        print(f"AVISO: {rid} tiene pesos y detecciones exportadas; se ignora {p}")
        continue
    d = encontrados.setdefault(rid, {"run_id": rid, "tipo": "externo", "tamano": "r18",
                                     "entrenado_con": "ENA24 + iNaturalist", "degrees": 0,
                                     "semilla": int(g["semilla"]), "pesos": "detecciones exportadas (PaddleDetection)",
                                     "ruta": {}, "clases": None, "mejor_epoca": None, "map5095_libreria": None})
    if conj in d["ruta"]:
        print(f"AVISO: {rid} tiene dos archivos de detecciones de '{conj}'; se usa {d['ruta'][conj]} y se ignora {p}")
        continue
    d["ruta"][conj] = p
for d in encontrados.values():          # mejor época y mAP50-95 de PaddleDetection, de <RUN_ID>_best.json
    if d["tipo"] != "externo":
        continue
    jsons = sorted(glob.glob(f"/kaggle/input/**/{d['run_id']}_best.json", recursive=True))
    if not jsons:
        print(f"AVISO: no hay {d['run_id']}_best.json; no se podrán comprobar su mejor época ni su mAP de PaddleDetection.")
        continue
    if len(jsons) > 1:
        print(f"AVISO: hay {len(jsons)} archivos {d['run_id']}_best.json; se usa {jsons[0]}")
    b = json.loads(Path(jsons[0]).read_text())
    d["mejor_epoca"], d["map5095_libreria"] = int(b["epoca"]), float(b["mAP50_95_paddle"])

encontrados["coco_n_sinentrenar"] = {"run_id": "coco_n_sinentrenar", "tipo": "coco", "tamano": "nano",
                                     "entrenado_con": "nada (pesos COCO, clase bear)", "degrees": 0,
                                     "semilla": -1, "pesos": "COCO", "ruta": COCO_PESOS, "clases": [COCO_CLASE]}

if INCLUIR is None:
    MODELOS = list(encontrados.values())
else:
    MODELOS = [encontrados[r] for r in INCLUIR if r in encontrados]
    for r in INCLUIR:
        if r not in encontrados:
            print(f"AVISO: '{r}' está en INCLUIR pero no hay pesos ni detecciones con ese nombre; se omite.")
ORDEN_TIPO = {"coco": 0, "yolo26": 1, "ablacion": 2, "externo": 3}
ORDEN_TAM = {"nano": 0, "small": 1, "medium": 2, "r18": 3}
MODELOS.sort(key=lambda d: (ORDEN_TIPO[d["tipo"]], ORDEN_TAM[d["tamano"]], d["run_id"]))
assert MODELOS, "No hay ningún modelo para evaluar."

if ignorados:
    print("Archivos con nombre no reconocido (no se usan):")
    for p in ignorados:
        print("  ", p)
print(f"Imágenes: {DATASET}")
print(f"{len(MODELOS)} modelos:")
for d in MODELOS:
    if d["tipo"] == "externo":
        archivos = ", ".join(f"{c}: {r}" for c, r in sorted(d["ruta"].items()))
        extra = (f" | mejor época {d['mejor_epoca']}, mAP50-95 de PaddleDetection {d['map5095_libreria']:.4f}"
                 if d["mejor_epoca"] is not None else " | sin best.json")
        print(f"   {d['run_id']:<22} {d['tipo']:<9} RT-DETRv3-R18 semilla={d['semilla']:<3} {archivos}{extra}")
    else:
        print(f"   {d['run_id']:<22} {d['tipo']:<9} {d['tamano']:<7} {d['entrenado_con']:<29} "
              f"semilla={d['semilla']:<3} {d['ruta']}")

Imágenes: /kaggle/input/datasets/josemartinvallejos/deteccionoso
7 modelos:
   e1_n_deg0_s0           yolo26    nano    ENA24 + iNaturalist           semilla=0   /kaggle/input/datasets/josemartinvallejos/yolodeteccionoso/e1_n_deg0_s0/e1_n_deg0_s0_best.pt
   e1b_n_deg0_s1          yolo26    nano    ENA24 + iNaturalist           semilla=1   /kaggle/input/datasets/josemartinvallejos/yolodeteccionoso/e1b_n_deg0_s1/e1b_n_deg0_s1_best.pt
   e4_m_deg0_s0           yolo26    medium  ENA24 + iNaturalist           semilla=0   /kaggle/input/datasets/josemartinvallejos/yolodeteccionoso/e4_m_deg0_s0/e4_m_deg0_s0_best.pt
   e4b_m_deg0_s1          yolo26    medium  ENA24 + iNaturalist           semilla=1   /kaggle/input/datasets/josemartinvallejos/yolodeteccionoso/e4b_m_deg0_s1/e4b_m_deg0_s1_best.pt
   e7_m_deg0_s0           yolo26    medium  ENA24 + iNaturalist           semilla=0   /kaggle/input/datasets/josemartinvallejos/yolodeteccionoso/e7_m_deg0_s0/e7_m_deg0_s0_best.pt
   e6_rtdetrv3r18_s0     

## 3. Imágenes y etiquetas

Para cada imagen: fuente (por el prefijo del nombre), cajas reales (desde `labels/`), si tiene oso, si es nocturna, y su ancho y alto tal como la leen Ultralytics y PaddleDetection (con la orientación EXIF aplicada). Nocturna se decide con el mismo criterio de `describir_dataset.py`: imagen reducida a 64×64 cuya diferencia media entre canales de color es menor que 2 (es decir, casi sin color, típico del infrarrojo).

Los conteos se comparan con los de `informe_preparacion_datos.md`. Si alguno no coincide, el cuaderno lo avisa. Las nocturnas se comparan solo en ENA24, porque `dividir_dataset.py` marcó todas las de iNaturalist como diurnas; las fotos de iNaturalist en blanco y negro se informan aparte y no se usan en las particiones de día y noche.

Para pycocotools, las cajas reales se pasan a píxeles con la misma conversión que `rtdetrv3_entrenar.ipynb` (`yolo_a_coco`): se recortan al borde de la foto y se descartan las que quedan sin área. Así, para RT-DETRv3, pycocotools ve exactamente las cajas reales que vio PaddleDetection.

In [4]:
import contextlib, io
import numpy as np
import pandas as pd
from PIL import Image, ImageChops, ImageStat
from pycocotools.coco import COCO

ESPERADO = {
    "val":  {"imagenes": 598, "positivas": 326, "cajas": 351, "ena24": 338, "inat": 260, "nocturnas": 224,
             "ena24_pos_dia": 52, "ena24_pos_noche": 104, "ena24_neg_dia": 62, "ena24_neg_noche": 120,
             "inat_pos": 170, "inat_neg": 90},
    "test": {"imagenes": 578, "positivas": 313, "cajas": 349, "ena24": 318, "inat": 260, "nocturnas": 197,
             "ena24_pos_dia": 59, "ena24_pos_noche": 86, "ena24_neg_dia": 62, "ena24_neg_noche": 111,
             "inat_pos": 168, "inat_neg": 92},
}
GIRA = {5, 6, 7, 8}   # orientaciones EXIF con las que OpenCV gira la foto e intercambia ancho y alto

def es_nocturna(ruta):
    """Mismo criterio que describir_dataset.py (escala_grises)."""
    with Image.open(ruta) as img:
        try:
            img.draft("RGB", (400, 400))
        except Exception:
            pass
        peq = img.convert("RGB").resize((64, 64))
    r, g, b = peq.split()
    dif = (ImageStat.Stat(ImageChops.difference(r, g)).mean[0]
           + ImageStat.Stat(ImageChops.difference(g, b)).mean[0]) / 2
    return round(dif, 2) < 2.0

def tamano_visto(ruta):
    """Ancho, alto y orientación EXIF de la foto, como la leen Ultralytics y PaddleDetection (con el giro aplicado)."""
    with Image.open(ruta) as im:
        W, H = im.size                      # solo lee la cabecera
        orient = im.getexif().get(0x0112, 1)
    return (H, W, orient) if orient in GIRA else (W, H, orient)

def fuente_de(nombre):
    if nombre.startswith("ena24_"):
        return "ena24"
    if nombre.startswith(("inat_", "inatneg_")):
        return "inat"
    return "otra"

def leer_cajas(ruta_txt):
    """Cajas YOLO (cx, cy, w, h normalizadas) convertidas a esquinas (x1, y1, x2, y2)."""
    if not ruta_txt.exists():
        return np.zeros((0, 4))
    filas = [l.split() for l in ruta_txt.read_text().splitlines() if l.strip()]
    if not filas:
        return np.zeros((0, 4))
    a = np.array([[float(v) for v in f[1:5]] for f in filas])
    return np.stack([a[:, 0] - a[:, 2] / 2, a[:, 1] - a[:, 3] / 2,
                     a[:, 0] + a[:, 2] / 2, a[:, 1] + a[:, 3] / 2], axis=1)

def tabla_imagenes(conjunto):
    filas, cajas = [], {}
    for ruta in sorted((DATASET / "images" / conjunto).glob("*.jpg")):
        gt = leer_cajas(DATASET / "labels" / conjunto / (ruta.stem + ".txt"))
        cajas[ruta.name] = gt
        W, H, orient = tamano_visto(ruta)
        filas.append({"conjunto": conjunto, "archivo": ruta.name, "ruta": str(ruta),
                      "fuente": fuente_de(ruta.name), "n_cajas": len(gt), "positiva": len(gt) > 0,
                      "nocturna": es_nocturna(ruta), "ancho": W, "alto": H, "orientacion_exif": orient})
    return pd.DataFrame(filas), cajas

def comprobar(conjunto, df):
    e = df[df.fuente == "ena24"]
    i = df[df.fuente == "inat"]
    obtenido = {
        "imagenes": len(df), "positivas": int(df.positiva.sum()), "cajas": int(df.n_cajas.sum()),
        "ena24": len(e), "inat": len(i), "nocturnas": int(e.nocturna.sum()),
        "ena24_pos_dia": int((e.positiva & ~e.nocturna).sum()), "ena24_pos_noche": int((e.positiva & e.nocturna).sum()),
        "ena24_neg_dia": int((~e.positiva & ~e.nocturna).sum()), "ena24_neg_noche": int((~e.positiva & e.nocturna).sum()),
        "inat_pos": int(i.positiva.sum()), "inat_neg": int((~i.positiva).sum()),
    }
    ok = True
    for k, v in obtenido.items():
        esp = ESPERADO[conjunto][k]
        marca = "" if v == esp else f"   <-- ATENCIÓN, se esperaba {esp}"
        ok &= v == esp
        print(f"   {k:<16} {v:>5}{marca}")
    # las listas por fuente deben contener exactamente las imágenes de cada fuente
    for fuente in ("ena24", "inat"):
        lista = DATASET / f"{conjunto}_{fuente}.txt"
        if lista.exists():
            en_lista = {Path(l.strip()).name for l in lista.read_text(encoding="utf-8").splitlines() if l.strip()}
            iguales = en_lista == set(df[df.fuente == fuente].archivo)
            ok &= iguales
            print(f"   lista {lista.name:<15} {'coincide' if iguales else 'NO COINCIDE con el prefijo del nombre'}")
    print(f"   (informativo) fotos de iNaturalist en blanco y negro: {int(i.nocturna.sum())}; "
          "no entran en ninguna partición de día o noche")
    print("   Todo coincide." if ok else "   Hay diferencias: revisar antes de interpretar los resultados.")

def silencio():
    """pycocotools imprime mucho; se descarta lo que escribe en pantalla."""
    return contextlib.redirect_stdout(io.StringIO())

def gt_coco(conj):
    """Cajas reales en formato COCO (píxeles), con la misma conversión que yolo_a_coco de rtdetrv3_entrenar.ipynb."""
    imagenes, anotaciones, recortadas, descartadas = [], [], 0, 0
    for img_id, r in enumerate(IMGS[conj].itertuples(), start=1):
        W, H = int(r.ancho), int(r.alto)
        imagenes.append({"id": img_id, "file_name": r.archivo, "width": W, "height": H})
        for x1, y1, x2, y2 in CAJAS[conj][r.archivo]:
            c = (float(x1 * W), float(y1 * H), float(x2 * W), float(y2 * H))
            rc = (max(0.0, c[0]), max(0.0, c[1]), min(float(W), c[2]), min(float(H), c[3]))
            recortadas += rc != c
            if rc[2] - rc[0] <= 0 or rc[3] - rc[1] <= 0:
                descartadas += 1
                continue
            anotaciones.append({"id": len(anotaciones) + 1, "image_id": img_id, "category_id": 1, "iscrowd": 0,
                                "bbox": [rc[0], rc[1], rc[2] - rc[0], rc[3] - rc[1]],
                                "area": (rc[2] - rc[0]) * (rc[3] - rc[1])})
    gt = COCO()
    gt.dataset = {"images": imagenes, "annotations": anotaciones, "categories": [{"id": 1, "name": "oso"}]}
    with silencio():
        gt.createIndex()
    return gt, recortadas, descartadas

CONJUNTOS = ["val", "test"] if EVALUAR_PRUEBA else ["val"]
IMGS, CAJAS, GT_COCO, ID_IMAGEN = {}, {}, {}, {}
for conj in CONJUNTOS:
    IMGS[conj], CAJAS[conj] = tabla_imagenes(conj)
    print(f"Conjunto '{conj}':")
    comprobar(conj, IMGS[conj])
    GT_COCO[conj], rec, desc = gt_coco(conj)
    ID_IMAGEN[conj] = {im["file_name"]: im["id"] for im in GT_COCO[conj].dataset["images"]}
    n_anot, n_exif = len(GT_COCO[conj].dataset["annotations"]), int((IMGS[conj].orientacion_exif != 1).sum())
    ok = n_anot == ESPERADO[conj]["cajas"] and desc == 0
    print(f"   Cajas reales para pycocotools: {n_anot} ({rec} recortadas al borde, {desc} descartadas); "
          f"fotos con orientación EXIF: {n_exif} -> {'bien' if ok else 'REVISAR'}")

Conjunto 'val':
   imagenes           598
   positivas          326
   cajas              351
   ena24              338
   inat               260
   nocturnas          224
   ena24_pos_dia       52
   ena24_pos_noche    104
   ena24_neg_dia       62
   ena24_neg_noche    120
   inat_pos           170
   inat_neg            90
   lista val_ena24.txt   coincide
   lista val_inat.txt    coincide
   (informativo) fotos de iNaturalist en blanco y negro: 9; no entran en ninguna partición de día o noche
   Todo coincide.
   Cajas reales para pycocotools: 351 (0 recortadas al borde, 0 descartadas); fotos con orientación EXIF: 0 -> bien


## 4. Detecciones de cada modelo

Cada modelo de Ultralytics procesa las imágenes una por una (lote de 1), con confianza mínima `CONF_MIN`. Los modelos de tipo `externo` (RT-DETRv3) no se ejecutan aquí: se leen sus detecciones exportadas, que salieron de la evaluación de PaddleDetection (cada foto reescalada a 640 × 640, como evalúa RT-DETRv3), y se comprueba que los nombres de archivo, las confianzas y las coordenadas tengan sentido. Después cada detección se marca como acierto o no: se recorren de mayor a menor confianza y cada una se asigna a la caja real libre con la que más se solapa, si ese solapamiento es de al menos `IOU_MIN`. Recorrerlas por confianza es lo que hace que el conteo sea exacto para cualquier umbral: lo que pasa por encima de un umbral no depende de lo que hay por debajo.

**Modelo COCO.** Se le pasa `classes=[COCO_CLASE]`, de modo que solo quedan sus detecciones de la clase `bear`. Verificado en el código de Ultralytics 8.4.138: con la cabeza end2end de YOLO26, el predictor sube a 300 como mínimo los candidatos que conserva la cabeza (`engine/predictor.py`, `setup_model`), la cabeza elige los 300 pares de candidato y clase con mayor puntaje entre las 80 clases (`nn/modules/head.py`, `get_topk_index`) y el filtro de clase se aplica después (`utils/nms.py`). Consecuencia: una respuesta débil de `bear` en una foto con otros objetos de puntaje más alto puede quedar fuera de esos 300. Afecta solo a confianzas bajas.

In [5]:
import time
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"

LISTAS = Path("/kaggle/working/listas")
LISTAS.mkdir(parents=True, exist_ok=True)

def predecir(modelo, df, conj, clases=None):
    # Se pasa un archivo de texto con las rutas, no una lista de Python. Con una lista, Ultralytics
    # abre todas las imágenes a la vez y las procesa en un solo lote, lo que agota la memoria de la GPU.
    # Con el archivo de texto lee una imagen por vez (lote de 1), igual que la aplicación final.
    lista = LISTAS / f"{conj}.txt"
    lista.write_text("\n".join(df["ruta"]) + "\n", encoding="utf-8")
    trozos, vistas = [], 0
    for r in modelo.predict(source=str(lista), batch=1, conf=CONF_MIN, imgsz=640, device=DEVICE,
                            classes=clases, stream=True, verbose=False):
        vistas += 1
        n = len(r.boxes)
        if n == 0:
            continue
        conf = r.boxes.conf.cpu().numpy()
        xy = r.boxes.xyxyn.cpu().numpy()
        trozos.append(pd.DataFrame({"archivo": [Path(r.path).name] * n, "conf": conf,
                                    "x1": xy[:, 0], "y1": xy[:, 1], "x2": xy[:, 2], "y2": xy[:, 3]}))
    assert vistas == len(df), f"Se procesaron {vistas} imágenes de {len(df)}"
    if not trozos:
        return pd.DataFrame(columns=["archivo", "conf", "x1", "y1", "x2", "y2"])
    return pd.concat(trozos, ignore_index=True)

def leer_externo(d, conj):
    """Detecciones exportadas (archivo, conf, x1, y1, x2, y2 normalizadas) de un modelo que no es de Ultralytics."""
    ruta = d["ruta"].get(conj)
    if ruta is None:
        raise SystemExit(f"Falta {d['run_id']}_detecciones_{conj}.csv.gz en los datos adjuntos.")
    p = pd.read_csv(ruta)
    faltan = {"archivo", "conf", "x1", "y1", "x2", "y2"} - set(p.columns)
    assert not faltan, f"{ruta.name}: faltan las columnas {faltan}"
    p = p[["archivo", "conf", "x1", "y1", "x2", "y2"]].copy()
    p["archivo"] = p["archivo"].astype(str)
    ajenos = set(p.archivo) - set(IMGS[conj].archivo)
    assert not ajenos, f"{ruta.name}: {len(ajenos)} archivos que no están en '{conj}', p. ej. {sorted(ajenos)[:3]}"
    assert p.conf.between(0, 1).all(), f"{ruta.name}: confianzas fuera de [0, 1]"
    bajas = int((p.conf < CONF_MIN).sum())
    p = p[p.conf >= CONF_MIN].reset_index(drop=True)
    xy = p[["x1", "y1", "x2", "y2"]].to_numpy(dtype=float)
    assert ((xy >= -1e-6) & (xy <= 1 + 1e-6)).all(), f"{ruta.name}: coordenadas fuera de [0, 1]"
    assert ((p.x2 >= p.x1) & (p.y2 >= p.y1)).all(), f"{ruta.name}: cajas con x2 < x1 o y2 < y1"
    por_img = p.groupby("archivo").size()
    print(f"   {ruta.name}: {len(p)} detecciones en {len(por_img)} de {len(IMGS[conj])} imágenes, "
          f"como máximo {por_img.max()} por imagen ({bajas} con confianza < {CONF_MIN} descartadas)")
    return p

def iou_matriz(a, b):
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-12)

def emparejar(pred, cajas, iou_min=None):
    """Marca cada detección como acierto (True) o no, con el solapamiento mínimo iou_min (por defecto IOU_MIN)."""
    iou_min = IOU_MIN if iou_min is None else iou_min
    tp = np.zeros(len(pred), dtype=bool)
    conf = pred["conf"].to_numpy(dtype=float)
    xy = pred[["x1", "y1", "x2", "y2"]].to_numpy(dtype=float)
    for nombre, idx in pred.groupby("archivo").indices.items():
        gt = cajas.get(nombre)
        if gt is None or len(gt) == 0:
            continue
        orden = idx[np.argsort(-conf[idx], kind="stable")]
        ious = iou_matriz(xy[orden], gt)
        libre = np.ones(len(gt), dtype=bool)
        for k, fila in enumerate(ious):
            fila = np.where(libre, fila, -1.0)
            j = int(fila.argmax())
            if fila[j] >= iou_min:
                libre[j] = False
                tp[orden[k]] = True
                if not libre.any():   # sin cajas libres, las detecciones que quedan en la foto no pueden acertar
                    break
    return tp

PRED = {}
for d in MODELOS:
    if d["tipo"] == "externo":
        for conj in CONJUNTOS:
            t0 = time.time()
            p = leer_externo(d, conj)
            p["tp"] = emparejar(p, CAJAS[conj])
            PRED[(d["run_id"], conj)] = p
            print(f"{d['run_id']:<22} {conj:<5} {len(p):>7} detecciones, {int(p.tp.sum()):>4} aciertos "
                  f"({time.time() - t0:.0f} s, detecciones exportadas)")
        continue
    modelo = YOLO(str(d["ruta"]))
    if d["clases"] is not None:
        nombres = [modelo.names[c] for c in d["clases"]]
        assert nombres == ["bear"], f"En {d['ruta']} la clase {d['clases']} es {nombres}, no 'bear'"
        print(f"{d['run_id']}: {len(modelo.names)} clases; se usa solo {d['clases']} = {nombres}")
    for conj in CONJUNTOS:
        t0 = time.time()
        p = predecir(modelo, IMGS[conj], conj, clases=d["clases"])
        p["tp"] = emparejar(p, CAJAS[conj])
        PRED[(d["run_id"], conj)] = p
        print(f"{d['run_id']:<22} {conj:<5} {len(p):>7} detecciones, {int(p.tp.sum()):>4} aciertos "
              f"({time.time() - t0:.0f} s)")
    del modelo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

e1_n_deg0_s0           val      1018 detecciones,  335 aciertos (21 s)
e1b_n_deg0_s1          val      1076 detecciones,  337 aciertos (15 s)
e4_m_deg0_s0           val      1201 detecciones,  337 aciertos (22 s)
e4b_m_deg0_s1          val      1340 detecciones,  340 aciertos (22 s)
e7_m_deg0_s0           val      1393 detecciones,  340 aciertos (22 s)
   e6_rtdetrv3r18_s0_detecciones_val.csv: 156160 detecciones en 598 de 598 imágenes, como máximo 300 por imagen (0 con confianza < 0.001 descartadas)
e6_rtdetrv3r18_s0      val    156160 detecciones,  351 aciertos (0 s, detecciones exportadas)
   e6b_rtdetrv3r18_s1_detecciones_val.csv: 178231 detecciones en 598 de 598 imágenes, como máximo 300 por imagen (0 con confianza < 0.001 descartadas)
e6b_rtdetrv3r18_s1     val    178231 detecciones,  351 aciertos (0 s, detecciones exportadas)


## 5. Umbral y métricas

Para cada modelo:
1. Umbral: la confianza más alta con la que el recall de cajas en validación (partición `UMBRAL_SOBRE`) sigue siendo al menos `RECALL_OBJETIVO`. Si el modelo no llega a ese recall ni con la confianza mínima, el umbral queda vacío (inalcanzable) y sus métricas también; la regla B se aplica después en local con `calcular_umbral.py`.
2. Con ese umbral, en cada partición: precisión, recall y F1 de cajas; recall por imagen; falsas alarmas.

Los porcentajes llevan un intervalo de confianza del 95 % (método de Wilson), porque con 100 a 300 imágenes por partición el margen es de varios puntos, como advierten los criterios (apartado 1.0).

Control de calidad: con las mismas detecciones se calcula mAP50 usando la función de Ultralytics (`compute_ap`) y se compara con el que reportó la validación de cada modelo. Deberían diferir poco: la validación de Ultralytics prepara las imágenes en lotes y empareja las cajas de otra forma. Medido en validación el 30 de septiembre de 2026 con los seis modelos de YOLO26: hasta 0.007 en global y en ENA24, con signo variable, y hasta 0.015 en iNaturalist, siempre por debajo del valor de Ultralytics. Causa probable, no verificada: la validación de Ultralytics procesa las fotos en lotes de 16 agrupados por forma y les añade un borde extra, mientras que este cuaderno procesa cada foto sola, como lo hará la aplicación; en iNaturalist, con fotos horizontales y verticales mezcladas, ese borde cambia más. Diferencias mucho mayores que estas sí indicarían un problema. Las referencias de la ablación tienen tres decimales (`experimentos.md`); el modelo COCO y RT-DETRv3 no tienen referencia de mAP50 (el control de RT-DETRv3 está en la sección 5b).

In [6]:
import math
from ultralytics.utils.metrics import compute_ap

REF_MAP50 = {  # mAP50 de la celda 5 del cuaderno de entrenamiento (validación de Ultralytics)
    "e1_n_deg0_s0":   {"global": 0.9318, "ena24": 0.9550, "inat": 0.9076},
    "e1b_n_deg0_s1":  {"global": 0.9285, "ena24": 0.9325, "inat": 0.9220},
    "e3_s_deg0_s0":   {"global": 0.9352, "ena24": 0.9457, "inat": 0.9198},
    "e3b_s_deg0_s1":  {"global": 0.9409, "ena24": 0.9574, "inat": 0.9302},
    "e4_m_deg0_s0":   {"global": 0.9330, "ena24": 0.9573, "inat": 0.9118},
    "e4b_m_deg0_s1":  {"global": 0.9363, "ena24": 0.9585, "inat": 0.9165},
    "a1_n_ena24_s0":  {"global": 0.743, "ena24": 0.942, "inat": 0.532},   # experimentos.md, tres decimales
    "a1b_n_ena24_s1": {"global": 0.745, "ena24": 0.932, "inat": 0.564},
    "a2_n_inat_s0":   {"global": 0.791, "ena24": 0.637, "inat": 0.914},
}
PARTICIONES = ["global", "ena24", "inat", "ena24_dia", "ena24_noche"]

def mascara(df, particion):
    if particion == "global":
        return np.ones(len(df), dtype=bool)
    if particion in ("ena24", "inat"):
        return (df.fuente == particion).values
    if particion == "ena24_dia":
        return ((df.fuente == "ena24") & ~df.nocturna).values
    if particion == "ena24_noche":
        return ((df.fuente == "ena24") & df.nocturna).values
    raise ValueError(particion)

def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    centro = (p + z * z / (2 * n)) / den
    margen = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, centro - margen), min(1.0, centro + margen))

def ap50(p, n_gt):
    if n_gt == 0 or len(p) == 0:
        return float("nan")
    orden = np.argsort(-p["conf"].values, kind="stable")
    tp = p["tp"].values[orden]
    ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
    return float(compute_ap(ctp / n_gt, ctp / np.maximum(ctp + cfp, 1))[0])

def umbral_para_recall(p, n_gt, objetivo):
    k = math.ceil(round(objetivo * n_gt, 9))
    conf_tp = np.sort(p.loc[p.tp, "conf"].values)[::-1]
    if n_gt == 0 or len(conf_tp) < k:
        return float("nan")  # inalcanzable: ni con la confianza mínima se llega al recall pedido
    return float(conf_tp[k - 1])

def metricas(run_id, conj, particion, t):
    df = IMGS[conj]
    m = mascara(df, particion)
    imgs = df[m]
    p = PRED[(run_id, conj)]
    p = p[p.archivo.isin(set(imgs.archivo))]
    n_gt = int(imgs.n_cajas.sum())
    pos = imgs.positiva.values
    n_pos, n_neg = int(pos.sum()), int((~pos).sum())
    fila = {"conjunto": conj, "particion": particion, "umbral": t, "n_cajas": n_gt,
            "n_pos": n_pos, "n_neg": n_neg, "ap50": ap50(p, n_gt)}
    if math.isnan(t):  # umbral inalcanzable: sin métricas a un umbral, para no mostrar ceros engañosos
        for c in ("n_detecciones", "aciertos", "P_caja", "R_caja", "F1_caja", "pos_detectadas", "recall_imagen",
                  "ri_ic_inf", "ri_ic_sup", "neg_con_alarma", "falsas_alarmas", "fa_ic_inf", "fa_ic_sup"):
            fila[c] = float("nan")
        return fila
    sel = p[p.conf >= t]
    n_pred, n_tp = len(sel), int(sel.tp.sum())
    P = n_tp / n_pred if n_pred else float("nan")
    R = n_tp / n_gt if n_gt else float("nan")
    F1 = 2 * P * R / (P + R) if (P + R) > 0 else float("nan")
    max_conf = imgs.archivo.map(p.groupby("archivo").conf.max()).fillna(0.0).values
    alarma = max_conf >= t
    k_pos, k_neg = int((alarma & pos).sum()), int((alarma & ~pos).sum())
    ri_lo, ri_hi = wilson(k_pos, n_pos)
    fa_lo, fa_hi = wilson(k_neg, n_neg)
    fila.update({"n_detecciones": n_pred, "aciertos": n_tp, "P_caja": P, "R_caja": R, "F1_caja": F1,
                 "pos_detectadas": k_pos, "recall_imagen": k_pos / n_pos if n_pos else float("nan"),
                 "ri_ic_inf": ri_lo, "ri_ic_sup": ri_hi,
                 "neg_con_alarma": k_neg, "falsas_alarmas": k_neg / n_neg if n_neg else float("nan"),
                 "fa_ic_inf": fa_lo, "fa_ic_sup": fa_hi})
    return fila

UMBRAL, UMBRAL_ALT, RECALL_MAX, filas = {}, {}, {}, []
otra = "global" if UMBRAL_SOBRE == "ena24" else "ena24"
for d in MODELOS:
    rid = d["run_id"]
    pv, dv = PRED[(rid, "val")], IMGS["val"]
    def n_y_p(part):
        imgs = dv[mascara(dv, part)]
        return pv[pv.archivo.isin(set(imgs.archivo))], int(imgs.n_cajas.sum())
    p_sobre, n_sobre = n_y_p(UMBRAL_SOBRE)
    UMBRAL[rid] = umbral_para_recall(p_sobre, n_sobre, RECALL_OBJETIVO)
    UMBRAL_ALT[rid] = umbral_para_recall(*n_y_p(otra), RECALL_OBJETIVO)
    RECALL_MAX[rid] = int(p_sobre.tp.sum()) / n_sobre if n_sobre else float("nan")
    for conj in CONJUNTOS:
        for part in PARTICIONES:
            f = metricas(rid, conj, part, UMBRAL[rid])
            f.update({"run_id": rid, "tipo": d["tipo"], "tamano": d["tamano"], "entrenado_con": d["entrenado_con"],
                      "semilla": d["semilla"], "umbral_sobre": UMBRAL_SOBRE})
            filas.append(f)
RES = pd.DataFrame(filas)

def txt_umbral(t):
    return "inalcanzable" if math.isnan(t) else f"{t:.3f}"

print(f"Umbral elegido sobre validación '{UMBRAL_SOBRE}' (y, solo como referencia, el que saldría sobre '{otra}'):")
for d in MODELOS:
    rid = d["run_id"]
    print(f"   {rid:<22} {txt_umbral(UMBRAL[rid]):<13} (sobre {otra}: {txt_umbral(UMBRAL_ALT[rid])}); "
          f"recall de cajas máximo en {UMBRAL_SOBRE} con confianza >= {CONF_MIN}: {RECALL_MAX[rid]:.3f}")

print("\nControl de calidad, mAP50 en validación: este cuaderno frente a Ultralytics")
for d in MODELOS:
    rid = d["run_id"]
    for part in ("global", "ena24", "inat"):
        mio = RES[(RES.run_id == rid) & (RES.conjunto == "val") & (RES.particion == part)].ap50.iloc[0]
        ref = REF_MAP50.get(rid, {}).get(part)
        dif = f"{mio - ref:+.4f}" if ref is not None else "sin referencia"
        print(f"   {rid:<22} {part:<7} {mio:.4f}  Ultralytics {ref if ref is not None else '-'}  diferencia {dif}")

Umbral elegido sobre validación 'ena24' (y, solo como referencia, el que saldría sobre 'global'):
   e1_n_deg0_s0           0.496         (sobre global: 0.223); recall de cajas máximo en ena24 con confianza >= 0.001: 0.975
   e1b_n_deg0_s1          0.611         (sobre global: 0.165); recall de cajas máximo en ena24 con confianza >= 0.001: 0.969
   e4_m_deg0_s0           0.498         (sobre global: 0.250); recall de cajas máximo en ena24 con confianza >= 0.001: 0.981
   e4b_m_deg0_s1          0.679         (sobre global: 0.354); recall de cajas máximo en ena24 con confianza >= 0.001: 0.988
   e7_m_deg0_s0           0.719         (sobre global: 0.384); recall de cajas máximo en ena24 con confianza >= 0.001: 0.981
   e6_rtdetrv3r18_s0      0.853         (sobre global: 0.850); recall de cajas máximo en ena24 con confianza >= 0.001: 1.000
   e6b_rtdetrv3r18_s1     0.847         (sobre global: 0.821); recall de cajas máximo en ena24 con confianza >= 0.001: 1.000

Control de calidad, mAP50 

## 5b. mAP común (pycocotools) y controles de cada librería

La cifra que se compara entre arquitecturas (pregunta 32 y regla de D25) es el mAP de COCO calculado por `pycocotools` sobre las detecciones de cada modelo, con sus parámetros estándar: umbrales de IoU de 0.50 a 0.95 de 0.05 en 0.05, interpolación de 101 puntos de recall, hasta 100 detecciones por foto (las de mayor confianza) y cajas de cualquier tamaño. Es el mismo evaluador que usó PaddleDetection para elegir la mejor época de RT-DETRv3. Las detecciones de cada modelo se pasan a píxeles de la foto original; las cajas reales son las de la sección 3. Se calcula sobre las cinco particiones; la regla de D25 usa la global.

Dos controles:
- **Cruce con el código del cuaderno**: el mismo mAP con el emparejamiento de la sección 4, repetido en los 10 umbrales de IoU, y `compute_ap` de Ultralytics. No coincide exactamente con pycocotools, porque integran la curva de forma distinta y este no corta en 100 detecciones por foto; las diferencias deberían ser de milésimas y parecidas entre modelos.
- **Frente a cada librería**: para YOLO26, el mAP50-95 de la validación de Ultralytics (tres decimales, `experimentos.md`), que se calcula de otra forma y no tiene por qué coincidir; para RT-DETRv3, el de PaddleDetection (`best.json`), que es el mismo evaluador sobre las mismas detecciones, con dos diferencias: el archivo exportado solo guarda las de confianza ≥ 0.001, y recorta al borde de la foto las cajas que se salen. PaddleDetection evaluó sin recortarlas. Ultralytics sí recorta las de YOLO26 (`utils/ops.py`, `scale_boxes` termina en `clip_boxes`; verificado en 8.4.138), así que con el recorte las dos arquitecturas se miden igual. El recorte solo puede subir el solapamiento con las cajas reales, que también están recortadas, por lo que la cifra común de RT-DETRv3 puede quedar unas milésimas por encima de la de PaddleDetection.

In [7]:
from pycocotools.cocoeval import COCOeval

UMBRALES_IOU = np.round(np.arange(0.50, 0.951, 0.05), 2)   # 0.50, 0.55, ..., 0.95, como COCO
REF_MAP5095 = {  # mAP50-95 de la validación de Ultralytics con best.pt (experimentos.md, tres decimales)
    "e1_n_deg0_s0":  {"global": 0.759, "ena24": 0.786, "inat": 0.732},
    "e1b_n_deg0_s1": {"global": 0.758, "ena24": 0.762, "inat": 0.758},
    "e3_s_deg0_s0":  {"global": 0.743, "ena24": 0.754, "inat": 0.738},
    "e3b_s_deg0_s1": {"global": 0.758, "ena24": 0.769, "inat": 0.754},
    "e4_m_deg0_s0":  {"global": 0.752, "ena24": 0.767, "inat": 0.746},
    "e4b_m_deg0_s1": {"global": 0.755, "ena24": 0.765, "inat": 0.755},
}

def dets_coco(p, conj):
    """Detecciones normalizadas -> lista COCO (x, y, ancho, alto en píxeles de la foto original)."""
    tam = IMGS[conj].set_index("archivo")
    W = p.archivo.map(tam.ancho).to_numpy(dtype=float)
    H = p.archivo.map(tam.alto).to_numpy(dtype=float)
    ids = p.archivo.map(ID_IMAGEN[conj]).to_numpy()
    xa, ya, xb, yb = (p[c].to_numpy(dtype=float) for c in ("x1", "y1", "x2", "y2"))
    return [{"image_id": int(i), "category_id": 1, "score": float(s),
             "bbox": [float(a * w), float(b * h), float((c - a) * w), float((e - b) * h)]}
            for i, s, a, b, c, e, w, h in zip(ids, p.conf.to_numpy(dtype=float), xa, ya, xb, yb, W, H)]

DT_COCO = {}
def eval_coco(run_id, conj, particion):
    """mAP50-95 y mAP50 de COCO con pycocotools, sobre las fotos de la partición."""
    clave = (run_id, conj)
    if clave not in DT_COCO:
        lista = dets_coco(PRED[clave], conj)
        with silencio():
            DT_COCO[clave] = GT_COCO[conj].loadRes(lista) if lista else None
    if DT_COCO[clave] is None:
        return float("nan"), float("nan")
    df = IMGS[conj]
    with silencio():
        ev = COCOeval(GT_COCO[conj], DT_COCO[clave], "bbox")
        ev.params.imgIds = [ID_IMAGEN[conj][a] for a in df.archivo[mascara(df, particion)]]
        ev.evaluate()
        ev.accumulate()
        ev.summarize()
    return float(ev.stats[0]), float(ev.stats[1])

TP_IOU = {}
def map_propio(run_id, conj, particion):
    """Cruce: mAP50-95 y mAP50 con el emparejamiento de la sección 4 y compute_ap de Ultralytics."""
    df = IMGS[conj]
    imgs = df[mascara(df, particion)]
    p = PRED[(run_id, conj)]
    sel = p.archivo.isin(set(imgs.archivo)).to_numpy()
    n_gt = int(imgs.n_cajas.sum())
    aps = [ap50(pd.DataFrame({"conf": p.conf.to_numpy()[sel], "tp": TP_IOU[(run_id, conj)][t][sel]}), n_gt)
           for t in UMBRALES_IOU]
    return float(np.mean(aps)), aps[0]

filas = []
for d in MODELOS:
    rid = d["run_id"]
    for conj in CONJUNTOS:
        p = PRED[(rid, conj)]
        TP_IOU[(rid, conj)] = {t: emparejar(p, CAJAS[conj], t) for t in UMBRALES_IOU}
        assert np.array_equal(TP_IOU[(rid, conj)][0.5], p.tp.to_numpy()), "El emparejamiento a 0.5 no coincide"
        for part in PARTICIONES:
            m5095, m50 = eval_coco(rid, conj, part)
            p5095, p50 = map_propio(rid, conj, part)
            if conj != "val":
                ref = None
            elif d["tipo"] == "externo":
                ref = d.get("map5095_libreria") if part == "global" else None
            else:
                ref = REF_MAP5095.get(rid, {}).get(part)
            filas.append({"run_id": rid, "tipo": d["tipo"], "semilla": d["semilla"], "conjunto": conj,
                          "particion": part, "mAP50_95_coco": m5095, "mAP50_coco": m50,
                          "mAP50_95_propio": p5095, "mAP50_propio": p50, "mAP50_95_libreria": ref})
MAPS = pd.DataFrame(filas)

print("mAP común (pycocotools), cruce con el código del cuaderno y cifra de cada librería")
print("(librería: Ultralytics para YOLO26, PaddleDetection para RT-DETRv3; '-' = sin referencia)")
for conj in CONJUNTOS:
    for part in ("global", "ena24", "inat"):
        sub = MAPS[(MAPS.conjunto == conj) & (MAPS.particion == part)]
        print(f"\n=== {conj} | {part} ===")
        print(pd.DataFrame({
            "modelo": sub.run_id,
            "mAP50-95 COCO": sub.mAP50_95_coco.round(4), "mAP50 COCO": sub.mAP50_coco.round(4),
            "mAP50-95 cuaderno": sub.mAP50_95_propio.round(4),
            "mAP50-95 librería": [("-" if v is None or pd.isna(v) else f"{v:.4f}") for v in sub.mAP50_95_libreria],
        }).to_string(index=False))

mAP común (pycocotools), cruce con el código del cuaderno y cifra de cada librería
(librería: Ultralytics para YOLO26, PaddleDetection para RT-DETRv3; '-' = sin referencia)

=== val | global ===
            modelo  mAP50-95 COCO  mAP50 COCO  mAP50-95 cuaderno mAP50-95 librería
      e1_n_deg0_s0         0.7494      0.9218             0.7519            0.7590
     e1b_n_deg0_s1         0.7388      0.9178             0.7412            0.7580
      e4_m_deg0_s0         0.7434      0.9274             0.7461            0.7520
     e4b_m_deg0_s1         0.7468      0.9291             0.7495            0.7550
      e7_m_deg0_s0         0.7478      0.9335             0.7506                 -
 e6_rtdetrv3r18_s0         0.8308      0.9616             0.8339            0.8307
e6b_rtdetrv3r18_s1         0.8270      0.9631             0.8302            0.8270

=== val | ena24 ===
            modelo  mAP50-95 COCO  mAP50 COCO  mAP50-95 cuaderno mAP50-95 librería
      e1_n_deg0_s0         0.7763    

## 6. Tablas de resultados

Las fracciones se muestran como aciertos/total, proporción y [intervalo del 95 %]. Un guion indica que el umbral es inalcanzable para ese modelo.

In [8]:
def frac(k, n, lo, hi):
    if not n or pd.isna(k):
        return "-"
    k, n = int(k), int(n)
    return f"{k}/{n} = {k / n:.2f} [{lo:.2f}-{hi:.2f}]"

def tabla(conj, particion):
    sub = RES[(RES.conjunto == conj) & (RES.particion == particion)]
    return pd.DataFrame({
        "modelo": sub.run_id, "umbral": [txt_umbral(t) for t in sub.umbral],
        "P caja": sub.P_caja.round(3), "R caja": sub.R_caja.round(3), "F1": sub.F1_caja.round(3),
        "recall por imagen": [frac(r.pos_detectadas, r.n_pos, r.ri_ic_inf, r.ri_ic_sup) for r in sub.itertuples()],
        "falsas alarmas": [frac(r.neg_con_alarma, r.n_neg, r.fa_ic_inf, r.fa_ic_sup) for r in sub.itertuples()],
    }).to_string(index=False)

TITULOS = {"ena24": "ENA24 (resultado principal)", "inat": "iNaturalist", "global": "Global",
           "ena24_dia": "ENA24 de día", "ena24_noche": "ENA24 de noche"}
for conj in CONJUNTOS:
    for part in PARTICIONES:
        print(f"\n=== {conj} | {TITULOS[part]} ===")
        print(tabla(conj, part))

print("\nMetas (criterios, apartado 1.1): recall por imagen >= 0.95; falsas alarmas < 0.10 por fuente; "
      "precisión de cajas >= 0.70.")


=== val | Global ===
            modelo umbral  P caja  R caja    F1          recall por imagen           falsas alarmas
      e1_n_deg0_s0  0.496   0.950   0.863 0.904 297/326 = 0.91 [0.88-0.94] 6/272 = 0.02 [0.01-0.05]
     e1b_n_deg0_s1  0.611   0.948   0.823 0.881 287/326 = 0.88 [0.84-0.91] 9/272 = 0.03 [0.02-0.06]
      e4_m_deg0_s0  0.498   0.955   0.855 0.902 289/326 = 0.89 [0.85-0.92] 5/272 = 0.02 [0.01-0.04]
     e4b_m_deg0_s1  0.679   0.957   0.821 0.883 280/326 = 0.86 [0.82-0.89] 6/272 = 0.02 [0.01-0.05]
      e7_m_deg0_s0  0.719   0.957   0.832 0.890 283/326 = 0.87 [0.83-0.90] 8/272 = 0.03 [0.01-0.06]
 e6_rtdetrv3r18_s0  0.853   0.972   0.889 0.929 301/326 = 0.92 [0.89-0.95] 4/272 = 0.01 [0.01-0.04]
e6b_rtdetrv3r18_s1  0.847   0.974   0.866 0.917 291/326 = 0.89 [0.85-0.92] 6/272 = 0.02 [0.01-0.05]

=== val | ENA24 (resultado principal) ===
            modelo umbral  P caja  R caja    F1          recall por imagen           falsas alarmas
      e1_n_deg0_s0  0.496   0.967  

## 7. Archivos para descargar

- `umbral_metricas.csv`: todas las cifras de las tablas, una fila por modelo, conjunto y partición.
- `umbral_imagenes.csv`: una fila por imagen con su fuente, si tiene oso, si es nocturna, su tamaño y la confianza máxima de cada modelo. Con este archivo se puede recalcular el recall por imagen y las falsas alarmas para cualquier umbral, sin GPU.
- `umbral_detecciones.csv.gz`: las detecciones con confianza de 0.01 o más, marcadas como acierto o no. Sirve para rehacer las curvas en local.
- `modelos_evaluados.csv`: qué modelos se evaluaron, con qué pesos o archivo de detecciones y qué clases.
- `map_comun.csv`: mAP50 y mAP50-95 de pycocotools, los del cruce con el código del cuaderno y la cifra de cada librería, por modelo, conjunto y partición (nuevo en la versión 3).

In [9]:
OUT = Path("/kaggle/working")
RES.to_csv(OUT / "umbral_metricas.csv", index=False)

imgs_out = []
for conj in CONJUNTOS:
    df = IMGS[conj].drop(columns=["ruta"]).copy()
    for d in MODELOS:
        p = PRED[(d["run_id"], conj)]
        df["maxconf_" + d["run_id"]] = df.archivo.map(p.groupby("archivo").conf.max()).fillna(0.0).round(4)
    imgs_out.append(df)
pd.concat(imgs_out, ignore_index=True).to_csv(OUT / "umbral_imagenes.csv", index=False)

det_out = []
for (rid, conj), p in PRED.items():
    q = p[p.conf >= 0.01].copy()
    q.insert(0, "conjunto", conj)
    q.insert(0, "run_id", rid)
    det_out.append(q)
pd.concat(det_out, ignore_index=True).round(4).to_csv(OUT / "umbral_detecciones.csv.gz", index=False)

pd.DataFrame([{k: (str(v) if k in ("ruta", "clases") else v) for k, v in d.items()} for d in MODELOS]) \
    .to_csv(OUT / "modelos_evaluados.csv", index=False)
MAPS.to_csv(OUT / "map_comun.csv", index=False)

for nombre in ("umbral_metricas.csv", "umbral_imagenes.csv", "umbral_detecciones.csv.gz", "modelos_evaluados.csv",
               "map_comun.csv"):
    print(f"   {nombre:<28} {(OUT / nombre).stat().st_size / 1e6:.2f} MB")

   umbral_metricas.csv          0.01 MB
   umbral_imagenes.csv          0.06 MB
   umbral_detecciones.csv.gz    0.27 MB
   modelos_evaluados.csv        0.00 MB
   map_comun.csv                0.00 MB


## 8. Controles y regla de D25

1. **E1 y E1b frente a septiembre.** Con el cuaderno versión 1, el 30 de septiembre, y repetido con la versión 2 el 2 de octubre: umbral A, fotos con oso detectadas y falsas alarmas en ENA24 de validación con ese umbral. Si E1 y E1b no repiten esas cifras, algo cambió (la imagen nueva de Kaggle o el cuaderno) y no se aplica la regla hasta explicarlo. E4 y E4b se comparan igual, como información. La regla B se comprueba después en local con `calcular_umbral.py`.
2. **RT-DETRv3 frente a PaddleDetection.** El mAP50-95 global de pycocotools sobre el archivo exportado debe quedar entre `TOL_PADDLE_BAJO` por debajo y `TOL_PADDLE_ALTO` por encima del de `best.json`. Se espera una diferencia pequeña y positiva, por el recorte al borde (sección 5b). Este control detecta errores gruesos, como un archivo de otra corrida o mal convertido; no distingue dos épocas de la meseta, que difieren en menos de 0.002.
3. **Regla de D25**, con los parámetros de la celda 0: condición 1 (mejor época) desde `best.json`, condición 2 (media de mAP50-95 global de pycocotools) desde la sección 5b.

In [10]:
REF_SEPTIEMBRE = {  # umbral A (tres decimales) y, con él, fotos con oso detectadas y falsas alarmas en ENA24 de validación
    "e1_n_deg0_s0":  (0.496, 144, 3),
    "e1b_n_deg0_s1": (0.611, 147, 5),
    "e4_m_deg0_s0":  (0.498, 144, 2),
    "e4b_m_deg0_s1": (0.679, 145, 2),
}
CONTROL = ["e1_n_deg0_s0", "e1b_n_deg0_s1"]
TOL_PADDLE_BAJO = 0.001   # el filtro de confianza >= 0.001 solo quita detecciones de la cola: efecto mínimo
TOL_PADDLE_ALTO = 0.010   # el recorte al borde puede subir el mAP unas milésimas

def fila_res(rid, part):
    return RES[(RES.run_id == rid) & (RES.conjunto == "val") & (RES.particion == part)].iloc[0]

def map5095(rid):
    return float(MAPS[(MAPS.run_id == rid) & (MAPS.conjunto == "val") & (MAPS.particion == "global")].mAP50_95_coco.iloc[0])

print("1) Frente a septiembre (umbral A y, con él, ENA24 de validación)")
iguales = {}
for rid, (t_ref, pos_ref, fa_ref) in REF_SEPTIEMBRE.items():
    if rid not in UMBRAL:
        print(f"   {rid:<22} no evaluado")
        continue
    t = UMBRAL[rid]
    if math.isnan(t):
        iguales[rid] = False
        print(f"   {rid:<22} umbral A inalcanzable (septiembre: {t_ref:.3f}) -> DISTINTO")
        continue
    f = fila_res(rid, "ena24")
    iguales[rid] = round(t, 3) == t_ref and int(f.pos_detectadas) == pos_ref and int(f.neg_con_alarma) == fa_ref
    print(f"   {rid:<22} umbral A {t:.3f} (sept. {t_ref:.3f}) | con oso {int(f.pos_detectadas)}/{int(f.n_pos)} "
          f"(sept. {pos_ref}) | falsas alarmas {int(f.neg_con_alarma)}/{int(f.n_neg)} (sept. {fa_ref}) -> "
          f"{'igual' if iguales[rid] else 'DISTINTO'}{'' if rid in CONTROL else ' (informativo)'}")
control_septiembre = all(iguales.get(r, False) for r in CONTROL)

print(f"\n2) RT-DETRv3: pycocotools sobre el archivo exportado frente a PaddleDetection "
      f"(se acepta de -{TOL_PADDLE_BAJO} a +{TOL_PADDLE_ALTO})")
coincide_paddle = {}
for d in MODELOS:
    if d["tipo"] != "externo":
        continue
    rid = d["run_id"]
    if d.get("map5095_libreria") is None:
        coincide_paddle[rid] = False
        print(f"   {rid:<22} sin best.json: no se puede comprobar")
        continue
    mio, ref = map5095(rid), d["map5095_libreria"]
    coincide_paddle[rid] = -TOL_PADDLE_BAJO <= mio - ref <= TOL_PADDLE_ALTO
    print(f"   {rid:<22} {mio:.4f} frente a {ref:.4f} (diferencia {mio - ref:+.4f}) -> "
          f"{'coincide' if coincide_paddle[rid] else 'REVISAR'}")

print("\n3) Regla de D25 (YOLO26m con 200 épocas)")
faltan = [r for r in GRUPO_RTDETR + GRUPO_YOLO26M if r not in UMBRAL]
if faltan:
    print(f"   Faltan modelos {faltan}: no se puede aplicar.")
else:
    epocas = {r: encontrados[r].get("mejor_epoca") for r in GRUPO_RTDETR}
    cond1 = all(e is not None and e <= EPOCA_MAX_D25 for e in epocas.values())
    print(f"   Condición 1, mejor época de RT-DETRv3 en la {EPOCA_MAX_D25} o antes: {epocas} -> "
          f"{'se cumple' if cond1 else 'no se cumple'}")
    for r in GRUPO_RTDETR + GRUPO_YOLO26M:
        print(f"      {r:<22} mAP50-95 global (pycocotools) {map5095(r):.4f}")
    media_r = float(np.mean([map5095(r) for r in GRUPO_RTDETR]))
    media_y = float(np.mean([map5095(r) for r in GRUPO_YOLO26M]))
    cond2 = media_r - media_y > MARGEN_D25
    print(f"   Condición 2: media RT-DETRv3 {media_r:.4f} - media YOLO26m {media_y:.4f} = {media_r - media_y:+.4f} "
          f"(hace falta más de {MARGEN_D25}) -> {'se cumple' if cond2 else 'no se cumple'}")
    if not (control_septiembre and all(coincide_paddle.get(r, False) for r in GRUPO_RTDETR)):
        print("   Algún control no pasó: NO aplicar la regla hasta explicarlo.")
    elif cond1 and cond2:
        print("   Resultado: se cumplen las dos condiciones -> entrenar YOLO26m con 200 épocas (D25).")
    else:
        print("   Resultado: no se cumplen las dos condiciones -> se compara a 100 épocas y se declara (D25).")

1) Frente a septiembre (umbral A y, con él, ENA24 de validación)
   e1_n_deg0_s0           umbral A 0.496 (sept. 0.496) | con oso 144/156 (sept. 144) | falsas alarmas 3/182 (sept. 3) -> igual
   e1b_n_deg0_s1          umbral A 0.611 (sept. 0.611) | con oso 147/156 (sept. 147) | falsas alarmas 5/182 (sept. 5) -> igual
   e4_m_deg0_s0           umbral A 0.498 (sept. 0.498) | con oso 144/156 (sept. 144) | falsas alarmas 2/182 (sept. 2) -> igual (informativo)
   e4b_m_deg0_s1          umbral A 0.679 (sept. 0.679) | con oso 145/156 (sept. 145) | falsas alarmas 2/182 (sept. 2) -> igual (informativo)

2) RT-DETRv3: pycocotools sobre el archivo exportado frente a PaddleDetection (se acepta de -0.001 a +0.01)
   e6_rtdetrv3r18_s0      0.8308 frente a 0.8307 (diferencia +0.0001) -> coincide
   e6b_rtdetrv3r18_s1     0.8270 frente a 0.8270 (diferencia +0.0001) -> coincide

3) Regla de D25 (YOLO26m con 200 épocas)
   Condición 1, mejor época de RT-DETRv3 en la 90 o antes: {'e6_rtdetrv3r18_s0': 85,

In [11]:
tipo_de = {d["run_id"]: d["tipo"] for d in MODELOS}
completas = []
for (rid, conj), p in PRED.items():
    if tipo_de[rid] == "externo":
        continue
    q = pd.DataFrame({"run_id": rid, "conjunto": conj, "archivo": p["archivo"].to_numpy()})
    for c in ("conf", "x1", "y1", "x2", "y2"):
        q[c] = p[c].to_numpy(dtype=float)
    completas.append(q)
    print(f"   {rid:<22} {conj:<5} {len(q):>6} detecciones, {int((q['conf'] < 0.01).sum()):>5} con confianza < 0.01")
assert completas, "No hay modelos de Ultralytics en esta corrida."
COMP = pd.concat(completas, ignore_index=True)
COMP.to_csv(OUT / "detecciones_completas.csv.gz", index=False, float_format="%.17g")

# Comprobación: el archivo, leído de vuelta, tiene que devolver exactamente los mismos números
vuelta = pd.read_csv(OUT / "detecciones_completas.csv.gz", float_precision="round_trip")
iguales = len(vuelta) == len(COMP) and all(
    np.array_equal(vuelta[c].to_numpy(), COMP[c].to_numpy()) for c in ("conf", "x1", "y1", "x2", "y2"))
print(f"detecciones_completas.csv.gz: {len(COMP)} filas, "
      f"{(OUT / 'detecciones_completas.csv.gz').stat().st_size / 1e6:.2f} MB; "
      f"leído de vuelta: {'idéntico' if iguales else 'DISTINTO, avisar antes de seguir'}")

   e1_n_deg0_s0           val     1018 detecciones,   519 con confianza < 0.01
   e1b_n_deg0_s1          val     1076 detecciones,   589 con confianza < 0.01
   e4_m_deg0_s0           val     1201 detecciones,   659 con confianza < 0.01
   e4b_m_deg0_s1          val     1340 detecciones,   791 con confianza < 0.01
   e7_m_deg0_s0           val     1393 detecciones,   842 con confianza < 0.01
detecciones_completas.csv.gz: 6028 filas, 0.30 MB; leído de vuelta: idéntico
